## BlobDatabricksSettings

### Description:
- Class BlobDatabricksSettings : Process of mounting Azure storage blob containers to Databrikcs at a specified location
### Note
- Note: If a container does not exist, it is automatically created.
- Note: マウントが既にされていた場合の挙動と既にアンマウントされているものを使用したときどういう挙動になるかを説明。

### BlobDatabricksSettings(storage_account_name, storage_account_key)

```
BlobDatabricksSettings(storage_account_name, storage_account_key)

Parameters
----------------
storage_account_name    :(string) Azure storage account name
storage_account_key     :(string) Azure storage account key

Returns
----------------
NA

```

### .mount_blob(blob_container_name, mount_point)

```
BlobDatabricksSettings.mount_blob(blob_container_name, mount_point)

Parameters
----------------
blob_container_name     :(string) Azure storage blob container name
mount_point             :(string) DBFS mount destination file path

Returns
----------------
NA

```

### .unmount_blob(mount_point)

```
BlobDatabricksSettings.unmount_blob(mount_point)

Parameters
----------------
mount_point             :(string) DBFS mount destination file path

Returns
----------------
NA

```

In [1]:
%pip install azure-storage-blob==12.16.0

Note: you may need to restart the kernel to use updated packages.


In [0]:
from azure.storage.blob import BlobServiceClient
from pathlib import Path

class BlobDatabricksSettings:

  def __init__(self, storage_account_name, storage_account_key):
    self.storage_account_name = storage_account_name
    self.storage_account_key = storage_account_key

  @staticmethod
  def _check_mount(mount_dir):
    return Path(mount_dir) in map(lambda x: Path(x.mountPoint), dbutils.fs.mounts())
  
  def mount_blob(self, blob_container_name, mount_point):
    try:
      # Blob Storageのマウント
      if not self._check_mount(mount_point):
          # BlobServiceClientの作成
          blob_service_client = BlobServiceClient(account_url=f"https://{self.storage_account_name}.blob.core.windows.net", credential=self.storage_account_key)

          # コンテナの存在確認
          container_client = blob_service_client.get_container_client(blob_container_name)
          if not container_client.exists():
              # コンテナの作成
              blob_service_client.create_container(blob_container_name)
              print("Created container: '{}'".format(blob_container_name))

          # Blob StorageをDatabricksにマウント
          source = f"wasbs://{blob_container_name}@{self.storage_account_name}.blob.core.windows.net"
          conf_key = f"fs.azure.account.key.{self.storage_account_name}.blob.core.windows.net"

          mounted = dbutils.fs.mount(
            source=source,
            mount_point=mount_point,
            extra_configs={conf_key: self.storage_account_key}
          )

    except Exception as e:
        raise e

  def unmount_blob(self, mount_point):
    # DatabricksのBlob Storageをアンマウント
    if self._check_mount(mount_point):
      unmounted = dbutils.fs.unmount(mount_point)
